# Descriptive phrase tagging of CPC samples
**What this notebook does:** select up to 50 DOCDB families per CPC prefix in a fixed time window; retrieve one English representative text per family; record literal matches to predefined phrases for smart glasses, eyewear, functions, audio and interaction. Report text coverage and phrase counts. Save SQL, raw texts, settings, rules and evidence.

**No relevance decisions:** phrase tags do not classify patents as relevant/irrelevant and do not recommend keeping, narrowing or removing CPC codes. A match can occur in unrelated contexts; a missing match can reflect different terminology. Audio/interaction matches are descriptive flags, not evidence of an Amplifon opportunity. This notebook does not discover keywords or validate official CPC definitions.

Use `sector_keywords.ipynb` for the main candidate collection, family/date/language checks and TF-IDF workflow. This notebook is optional. The historical file/function names are retained for compatibility; they do not imply validation.

Run on TIP and record the PATSTAT release. TEST is for development, PROD for final analysis. The default 2000–2022 window is configurable, not a guarantee of data completeness. Sampling is deterministic within a fixed snapshot. Families can occur under several prefixes: do not sum prefix counts as sector size.

In [ ]:
from pathlib import Path
import sys

# Works when launched from the repository root or project directory.
PROJECT_DIR = Path.cwd() if Path("cpc_validation.py").exists() else Path.cwd() / "project"
if not (PROJECT_DIR / "cpc_validation.py").exists():
    raise RuntimeError("Open this notebook from the repository root or project directory")
sys.path.insert(0, str(PROJECT_DIR.resolve()))

from epo.tipdata.patstat import PatstatClient
from cpc_validation import run_validation
import pandas as pd

%run "$PROJECT_DIR/key_words.ipynb"

ENV = "TEST"
DATASET_RELEASE = ""  # Required: copy the PATSTAT release identifier from TIP.
YEAR_MIN, YEAR_MAX = 2000, 2022
SAMPLE_SIZE = 50
SEED = 42


In [ ]:
if not DATASET_RELEASE.strip():
    raise ValueError("Set DATASET_RELEASE to the PATSTAT release identifier shown in TIP")

patstat = PatstatClient(env=ENV)
summaries, evidence, output_path = run_validation(
    patstat, CPC_CODES, env=ENV, dataset_release=DATASET_RELEASE,
    year_min=YEAR_MIN, year_max=YEAR_MAX,
    sample_size=SAMPLE_SIZE, seed=SEED,
    output_dir=PROJECT_DIR / "outputs" / "cpc_validation",
)
summary_df = pd.DataFrame(summaries)
display(summary_df)
print(f"Saved outputs: {output_path.resolve()}")


## Reading the output
`cpc_summary.csv` reports sampled/readable family counts, English text coverage, phrase-group counts and overlapping string prefixes. It contains no relevance scores, thresholds or recommendations.

`family_evidence.csv` records the representative text and these descriptive columns:
- `has_english_text`: non-empty supplied English text.
- `direct_phrase_present`: a predefined smart-glasses phrase occurs.
- `eyewear_phrase_present`: a predefined eyewear/head-worn phrase occurs.
- `function_phrase_present`: a predefined functional term occurs.
- `audio_theme` / `interaction_theme`: corresponding terms occur.
- `matched_terms`: exact normalised phrases that triggered each flag.

A zero count means no phrase match in the sample, not no relevant patents. No sampled families means no data for this environment/window. Missing text remains in the coverage denominator; coverage is undefined when the sample is empty. Prefixes can overlap. Old output directories may contain earlier recommendations; only new runs use descriptive tagging v2.

No CPC code or candidate family is excluded using these tags.